In [ ]:
#!import C:\Users\musharm\source\repos\performance_dynamic\src\benchmarks\gc\GC.Infrastructure\Notebooks\BenchmarkAnalysis_Fundamentals.ipynb

In [ ]:
var fragmentationFixPath = @"C:\Users\musharm\source\repos\performance\src\benchmarks\gc\GC.Infrastructure\Configurations\ASPNetBenchmarks\FragmentationFix_Data";
var dm = DataManager.CreateAspNetData(ML(fragmentationFixPath));

In [ ]:
dm

In [ ]:
static Metric<TraceGC> excessFragmentation = new Metric<TraceGC>((d => {
    Dictionary<CondemnedReasonGroup, int> condemnReasonsPerGC = new Dictionary<CondemnedReasonGroup, int>();
    d.GetCondemnedReasons(condemnReasonsPerGC);
    return condemnReasonsPerGC.ContainsKey(CondemnedReasonGroup.Fragmented_Ephemeral) ? 1 : 0; 
}), "IsExcessEphFrag", "#");
static Aggregation sumAggregation = new Aggregation((d => d.Sum()), "Sum", "#");
static Metric<IterationData> iterationExcessFragmentation = Metrics.Promote(excessFragmentation, sumAggregation);
static Metric<BenchmarkData> TotalExcessFragmentation = Metrics.Promote(iterationExcessFragmentation, Aggregation.Average);

### Existence of Issue in Fix

In [ ]:
// Num of GCs done.
// Individual GCs.
// Volatility within the Runs.
TableBenchmarks(dm, ML( TotalExcessFragmentation ), configFilter: new Filter( ML ( "datas_fix", "datas_nofix", "server_fix" )), compareInfo: new CompareInfo("datas_nofix", "datas_fix"), textPresenter: new MarkdownPresenter());

### Server Fix vs. DATAS Fix

In [ ]:
var t = TableBenchmarks(dm, ML( Metrics.B.AverageMaxHeapSize, Metrics.B.AverageRequestPerMSec, Metrics.B.AverageP50Latency),
 compareInfo: new CompareInfo("datas_nofix", "datas_fix"), textPresenter: new MarkdownPresenter() );
string.Join("\n", t[0]).Display();

In [ ]:
var t = TableBenchmarks(dm, ML( Metrics.B.AverageMaxHeapSize, Metrics.B.AverageRequestPerMSec, Metrics.B.AverageP50Latency),
 compareInfo: new CompareInfo("server_fix", "datas_fix"), textPresenter: new MarkdownPresenter() );

// Rename CompareInfo -> ConfigCompareInfo

### Server No Fix vs. DATAS No Fix

In [ ]:
TableBenchmarks(dm, ML( Metrics.B.AverageMaxHeapSize, Metrics.B.AverageRequestPerMSec, Metrics.B.AverageP50Latency), configFilter: new Filter( ML ( "server_nofix", "datas_nofix" )),
 compareInfo: new CompareInfo("server_nofix", "datas_nofix"), textPresenter: new MarkdownPresenter() );